# Etapa 1: Análise Exploratória e Carga de Dados

Realizou-se a carga dos dados brutos disponibilizados para o desafio. O objetivo principal consistiu em compreender a estrutura do conjunto de dados de treinamento, composto por características dos estabelecimentos (localização, atributos e categorias) e pela variável alvo `destaque` (1 para alta avaliação, 0 caso contrário).

## Dataset original

## Engenharia de Features

Para a predição de `destaque`, foi estruturado um pipeline de engenharia de features combinando técnicas geoespaciais, de processamento de linguagem natural (NLP) e de redução de dimensionalidade.

### 1. Tratamento da coluna `categories` (NLP e PCA)
- **Vetorização Semântica (Word Embeddings)**: Foram utilizados modelos pré-treinados do spaCy (`en_core_web_md`) para a geração dos vetores semânticos médios das categorias.
- **Similaridade de Cosseno**: Realizou-se o cálculo da similaridade de cosseno entre os vetores dos estabelecimentos e perfis de referência (`categories_cossine`).
- **Redução de Dimensionalidade (PCA)**: Após a aplicação de *One-Hot Encoding*, empregou-se o **PCA (Principal Component Analysis)** para reduzir as categorias a 3 componentes principais.

### 2. Tratamento da coluna `attributes`
- A estrutura JSON dos atributos originais (ex: `WiFi`, `NoiseLevel`) foi normalizada.
- Foi aplicado PCA para a extração das variáveis explicativas referentes às comodidades dos estabelecimentos.

### 3. Clusterização Geoespacial com DBSCAN
- **Clusterização**: Aplicou-se o algoritmo **DBSCAN** sobre as coordenadas de latitude e longitude para a identificação de agrupamentos por densidade geográfica na cidade de Toronto.
- O algoritmo converteu a localização física em *features* indicadoras de concentração espacial.

### 4. Análise de Sentimentos (Reviews)
- As avaliações textuais foram processadas via NLTK (`SentimentIntensityAnalyzer`), extraindo-se o *compound score* como indicador contínuo de sentimento.


## Tratamento coluna `categories`

- **Tratamento de Linhas Vazias**: As linhas da coluna `categories` que estavam vazias ou nulas foram tratadas para evitar problemas na análise posterior.

- **Separação das Categorias**: As strings da coluna `categories`, contendo múltiplas categorias separadas por vírgulas, foram divididas em listas de categorias individuais.

- **Explosão das Listas de Categorias**: Após a separação, a coluna foi "explodida", ou seja, cada categoria foi expandida em uma linha separada, mantendo as demais colunas constantes. Isso permitiu um tratamento individualizado de cada categoria.

- **Codificação das Categorias**: As categorias foram codificadas em colunas booleanas (one-hot encoding), onde cada categoria se tornou uma coluna no DataFrame, com valores `0` ou `1` indicando a presença ou ausência daquela categoria.

- **Cálculo de Embeddings**: A coluna `categories` foi transformada em embeddings utilizando técnicas de processamento de linguagem natural (Word Embeddings), permitindo que as categorias sejam representadas por vetores numéricos que capturam semânticas.

- **Análise de Similaridade**: A similaridade entre as categorias foi calculada usando a similaridade do cosseno entre os embeddings, permitindo identificar quão parecidas são as categorias entre si.

- **Redução de Dimensionalidade**: Os embeddings das categorias passaram por uma redução de dimensionalidade utilizando PCA (Principal Component Analysis), resultando em componentes principais que foram utilizados como features no modelo final.

- **Identificação de Categorias Populares**: Uma coluna adicional foi criada para identificar se uma categoria pertence ao conjunto das categorias mais populares, auxiliando na criação de features mais informativas.

## Tratamento da Coluna `attributes`

- **Extração e Normalização**: A coluna `attributes`, que continha informações estruturadas como JSON em formato de string, foi convertida em um DataFrame utilizando `json_normalize`. Isso permitiu que cada atributo presente na string fosse separado em colunas distintas para facilitar a análise.

- **Tratamento de Valores Nulos**: Após a normalização, as colunas que resultaram da extração de atributos tiveram seus valores nulos preenchidos com a string `"SEM_VALOR"`. Isso foi necessário para garantir que modelos preditivos futuros não enfrentassem problemas com dados ausentes.

- **Processamento de Atributos Textuais**: Alguns atributos textuais, como `WiFi`, `NoiseLevel`, e `RestaurantsAttire`, passaram por um tratamento adicional para remover prefixos indesejados (e.g., `'u'`) e outros caracteres que poderiam interferir na análise.

## Agrupamento DBSCAN

- **Objetivo do Agrupamento**: O algoritmo de agrupamento DBSCAN foi aplicado para identificar clusters naturais entre os estabelecimentos com base em suas características numéricas. Isso foi útil para detectar padrões de comportamento entre os estabelecimentos, como agrupamentos de estabelecimentos com características semelhantes.

- **Parâmetros do Algoritmo**: Os parâmetros `eps` e `min_samples` foram ajustados para identificar agrupamentos densos e ignorar pontos ruidosos. O valor de `eps` define a distância máxima entre dois pontos para que eles sejam considerados parte do mesmo cluster, enquanto `min_samples` é o número mínimo de pontos necessários para formar um cluster.

- **Interpretação dos Resultados**: Os clusters resultantes foram interpretados para entender se havia algum padrão específico entre os estabelecimentos. Essa análise foi fundamental para categorizar os estabelecimentos antes de aplicar os modelos preditivos.

## Análise dos Reviews dos Estabelecimentos

- **Coleta e Limpeza dos Reviews**: As avaliações textuais dos estabelecimentos foram coletadas e passaram por um processo de limpeza.

- **Geração de Embeddings**: Foi utilizado um modelo de linguagem para converter os reviews em embeddings, representações vetoriais que capturam o significado semântico dos textos. Esses embeddings foram utilizados como insumos para a análise de sentimentos e também para treinar os modelos preditivos.

- **Integração com Atributos**: As características extraídas dos reviews foram integradas com as outras características dos estabelecimentos, como as presentes na coluna `attributes`, formando um conjunto de dados rico e completo para o treinamento dos modelos preditivos.

## Considerações Finais

- **Preparação para Modelos Preditivos**: O tratamento das features preparou o dataset para o treinamento de modelos preditivos. Essas etapas foram essenciais para garantir a qualidade e a relevância dos dados, resultando em um modelo mais robusto e preciso.


## Dataset Tratado e Padronizado

Após a etapa de transformação, as variáveis foram padronizadas com o `StandardScaler` para posterior alimentação dos classificadores.

In [1]:
import carga_tratamento_dados as carga

df_treino, df_validacao = carga.carregar_dados()

display(f"Tamanho dataset treinamento: {len(df_treino)}")

display(df_treino.head())

INFO:carga_tratamento_dados:Carregando dados
INFO:carga_tratamento_dados:Tratando linhas com categoria vazia
INFO:carga_tratamento_dados:Seleção PCA para categorias
INFO:carga_tratamento_dados:Tratando categorias não populares usando similaridade do cosseno
100%|██████████| 17582/17582 [01:18<00:00, 225.14it/s]
INFO:carga_tratamento_dados:Adicionando coluna categoria popular
INFO:carga_tratamento_dados:Acrescentando coluna embedding para representar as categorias
100%|██████████| 17582/17582 [01:08<00:00, 254.82it/s]
INFO:carga_tratamento_dados:Tratando reviews
100%|██████████| 490963/490963 [09:55<00:00, 825.07it/s] 
INFO:carga_tratamento_dados:Padronizar dados


'Tamanho dataset treinamento: 14065'

,review_count,is_open,cat_pca_0,cat_pca_1,cat_pca_2,categories_cossine,popular_categories,categories_embedding,attr_pca_0,attr_pca_1,attr_pca_2,agr_lat_log_popular_0.0,agr_lat_log_popular_1.0,agr_lat_log_popular_2.0,agr_lat_log_nao_popular_0.0,avg_sentimento,destaque
business_id,,,,,,,,,,,,,,,,,
vHzWmPWHN4J1hRR3W3AMQg,1.190565,0.564871,1.272565,-1.570518,0.661632,0.105376,-0.25532,-0.434373,0.285242,0.156574,0.107292,2.408640,-0.04715,-0.016866,-0.816032,0.758168,0
15to24Q-otAHmto7FzsWRg,-0.400175,0.564871,-1.704222,-0.337352,-0.597444,0.206345,-0.25532,0.103852,0.285242,0.156574,0.107292,-0.415172,-0.04715,-0.016866,1.225442,1.111536,1
8aqKdf4G4AAir8k_Kdslvg,-0.151622,0.564871,-0.716522,-0.138046,-0.769267,-1.181509,-0.25532,0.642367,0.241080,-3.559565,-5.557528,-0.415172,-0.04715,-0.016866,-0.816032,-0.487166,0
uxU1vr5AhhkTQ83X0bpeyg,-0.400175,-1.770317,-0.718857,-0.138736,-0.773645,-0.939813,-0.25532,0.447907,0.285242,0.156574,0.107292,-0.415172,-0.04715,-0.016866,1.225442,0.757734,0
f702hTJoqdR34Jn23C7d1A,-0.400175,0.564871,-1.290275,-0.148232,1.331041,-0.428198,-0.25532,0.046060,-1.120791,-1.814593,1.589562,-0.415172,-0.04715,-0.016866,1.225442,-2.647098,0


## Treinamento dos Modelos

Após o pré-processamento, realizou-se o treinamento de diferentes arquiteturas de Machine Learning para otimização do **Mean F1-Score**.

### RandomForestClassifier
Classificador baseado em ensemble de árvores de decisão. Foi utilizado o `GridSearchCV` para o ajuste de hiperparâmetros via validação cruzada.

In [2]:
import treinamento_modelos as treinamento
from IPython.display import display, Markdown

score_treinamento, modelo_random_forest = treinamento.treinar_random_forest(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

INFO:root:Score RandomForestClassifier: 0.8500986922950634
INFO:root:Melhores parâmetros: {'class_weight': None, 'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100, 'random_state': 42}


## Score treinamento: 0.8500986922950634

INFO:root:F1-score dataset validação: 0.8563185417519679


## Score validação: 0.8563185417519679

### XGBoost Classifier
Modelo de *Gradient Boosting*. O treinamento foi conduzido com busca em grade para o ajuste de parâmetros de regularização e aprendizado.

In [3]:
score_treinamento, modelo_random_forest = treinamento.treinar_xgboost(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

INFO:root:Score XGBClassifier: 0.854771502605012
INFO:root:Melhores parâmetros: {'colsample_bytree': 0.9, 'eval_metric': 'logloss', 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 100, 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.9}


## Score treinamento: 0.854771502605012

INFO:root:F1-score dataset validação: 0.860734223888312


## Score validação: 0.860734223888312

### Multi-Layer Perceptron (MLP)
Rede neural artificial (MLP). Foram ajustados os parâmetros da camada oculta e a regularização via otimizador Adam.

In [4]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

score_treinamento, modelo_random_forest = treinamento.treinar_mlp(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

INFO:root:Score MLPClassifier: 0.8493062609031226
INFO:root:Melhores parâmetros: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (100, 100), 'learning_rate': 'adaptive', 'max_iter': 300, 'random_state': 42, 'solver': 'sgd'}


## Score treinamento: 0.8493062609031226

INFO:root:F1-score dataset validação: 0.8611503061643173


## Score validação: 0.8611503061643173

### Ensemble Final: Voting Classifier
Foi construído um `VotingClassifier` (método *Hard Voting*), combinando as predições do RandomForest, XGBoost e MLP em um modelo ensemble final.

In [5]:
score_treinamento, modelo_random_forest = treinamento.treinar_voting_classifier(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

INFO:root:F1-Score do VotingClassifier: 0.8814293281315402


## Score treinamento: 0.8814293281315402

INFO:root:F1-score dataset validação: 0.8620298936239859


## Score validação: 0.8620298936239859